In [6]:
from pathlib import Path
from typing import List, Any
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader, DirectoryLoader, GitLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders.excel import UnstructuredExcelLoader
from langchain_community.document_loaders import JSONLoader



def load_all_documents(data_dir: str) -> List[Any]:
    """
    Load all supported files from the diata directory and converst to LangChain
    Document Structure.

    Supported : PDF, TXT, CSV, Excel, Word, JSON, GitHub Repo
    """

    data_path = Path(data_dir).resolve()
    print(f"[DEBUG] Data path: {data_path}")
    documents = []

    # PDF files

    pdf_files = list((data_path / "pdf_file").glob("*.pdf"))

    for pdf_file in pdf_files:
        try: 
            loader = PyMuPDFLoader(str(pdf_file))
            loaded = loader.load()
            documents.extend(loaded)
        except Exception as e:
            print(f"[ERROR] failed to load {pdf_file}: {e}")


    return documents


In [7]:
load_all_documents(r"C:\Users\abirf\OneDrive\Desktop\AI Course\RAG\data_dir")

[DEBUG] Data path: C:\Users\abirf\OneDrive\Desktop\AI Course\RAG\data_dir


[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-07T09:01:21+00:00', 'source': 'C:\\Users\\abirf\\OneDrive\\Desktop\\AI Course\\RAG\\data_dir\\pdf_file\\CV_Md_Abir__AkerSolutions.pdf', 'file_path': 'C:\\Users\\abirf\\OneDrive\\Desktop\\AI Course\\RAG\\data_dir\\pdf_file\\CV_Md_Abir__AkerSolutions.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-10-07T09:01:21+00:00', 'trapped': '', 'modDate': 'D:20261007090121Z', 'creationDate': 'D:20261007090121Z', 'page': 0}, page_content='Md Abir Hossain\n+44 7507034017 | abirfarabi915@gmail.com | linkedin.com/in/abir-farabi | Github | Birmingham, UK\nSummary\nMSc graduate in Artificial Intelligence and Machine Learning from the University of Birmingham. Hands-on\nexperience building machine learning & AI models across multiple domains including generative AI, RAG,\ncomputer vision, data science, statistical analysis, ti

In [ ]:
def load_github_repos(repo_list_file: Path, clone_dir: Path) -> List[Any]:
    """Clone each repo listed in repo_list_file and load its README.md and requirements.txt."""
    documents: List[Any] = []
    wanted_files = {"README.md", "requirements.txt"}

    for line in repo_list_file.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):          # skip blanks and comments
            continue

        parts = line.split()
        url = parts[0]
        branch = parts[1] if len(parts) > 1 else "main"
        repo_name = url.rstrip("/").split("/")[-1].removesuffix(".git")

        try:
            loader = GitLoader(
                clone_url=url,
                repo_path=str(clone_dir / repo_name),
                branch=branch,
                file_filter=lambda p: Path(p).name in wanted_files,
            )
            loaded = loader.load()

            for doc in loaded:                         # tag so answers can cite the repo
                doc.metadata["source_type"] = "github"
                doc.metadata["repo"] = repo_name
            documents.extend(loaded)
        except Exception as e:
            print(f"[ERROR] failed to load repo {url}: {e}")

    return documents





In [11]:
data_path = Path(r"C:\Users\abirf\OneDrive\Desktop\AI Course\RAG\data_dir")

In [12]:
github_docs = load_github_repos(
    repo_list_file=data_path / "github" / "repos.txt",
    clone_dir=data_path / "repo_cache",
)
print(len(github_docs))

11


In [13]:
github_docs

[Document(metadata={'source': 'README.md', 'file_path': 'README.md', 'file_name': 'README.md', 'file_type': '.md', 'source_type': 'github', 'repo': 'Eye-disease-classification-with-PyTorch'}, page_content='# Eye Infection Classification (Bacterial vs Fungal vs Healthy)\r\n\r\nA deep learning classifier that distinguishes bacterial keratitis, fungal keratitis, and healthy eyes from clinical photographs, built with PyTorch and transfer learning.\r\n\r\n## Problem\r\n\r\nGiven a photograph of a patient\'s eye, classify it into one of three categories:\r\n- **Bacterial** infection\r\n- **Fungal** infection\r\n- **Healthy**\r\n\r\nThis is a 3-class image classification problem built on real clinical images, with the eventual goal of extending to fungal subtype classification (Aspergillus, Candida, Fusarium, Others) as a follow-up project.\r\n\r\n## Dataset\r\n\r\n- ~4,600 images total across three classes, sourced from patient photographs (multiple images per patient).\r\n- Fungal subtypes 